# Bronze → Silver

Notebook completo da camada Silver. A Bronze e somente leitura. As transformacoes abaixo aplicam os nomes em portugues, tipagem, deduplicacao, tratamento de dados invalidos, Forward Fill da PTAX e saneamento do Column Shift conforme o escopo.


In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

#Funcoes auxiliares reutilizadas nas transformacoes da camada Silver
def carga_mais_recente(df, chave="id"):
    """Mantem a versao mais recente de cada chave usando ingestion_datetime."""
    df = df.withColumn(chave, f.trim(f.col(chave)))
    w = Window.partitionBy(chave).orderBy(f.col("ingestion_datetime").desc())
    return (
        df.withColumn("_rn", f.row_number().over(w))
          .filter(f.col("_rn") == 1)
          .drop("_rn")
    )


def numero_decimal(coluna):
    """Converte texto decimal aceitando ponto ou virgula decimal e separadores mistos."""
    x = f.trim(coluna.cast("string"))
    x = f.regexp_replace(x, r"\s+", "")

    #Quando existem ponto e virgula no mesmo valor, o ultimo separador e tratado como separador decimal
    x = (
        f.when(x.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
               f.regexp_replace(f.regexp_replace(x, r"\.", ""), ",", "."))
         .when(x.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
               f.regexp_replace(x, ",", ""))
         .when(x.rlike(r"^-?\d+,\d+$"), f.regexp_replace(x, ",", "."))
         .otherwise(x)
    )
    return f.when(x.rlike(r"^-?\d+(\.\d+)?$"), x.cast("double"))


def numero_monetario(coluna):
    """Remove moeda/ruido e converte formatos como 1,234.56; 1.234,56; 1,234; 1.234."""
    x = f.lower(f.trim(coluna.cast("string")))
    x = f.when(
        x.isNull() | x.isin("", "unknown", "nao informado", "nao informado", "n/a", "null", "none"),
        f.lit(None)
    ).otherwise(x)
    x = f.regexp_replace(x, r"(?i)usd|brl|r\$|\$|\s", "")

    x = (
        f.when(x.rlike(r"^-?\d{1,3}(\.\d{3})+,\d+$"),
               f.regexp_replace(f.regexp_replace(x, r"\.", ""), ",", "."))
         .when(x.rlike(r"^-?\d{1,3}(,\d{3})+\.\d+$"),
               f.regexp_replace(x, ",", ""))
         .when(x.rlike(r"^-?\d{1,3}(,\d{3})+$"), f.regexp_replace(x, ",", ""))
         .when(x.rlike(r"^-?\d{1,3}(\.\d{3})+$"), f.regexp_replace(x, r"\.", ""))
         .when(x.rlike(r"^-?\d+,\d+$"), f.regexp_replace(x, ",", "."))
         .otherwise(x)
    )
    return f.when(x.rlike(r"^-?\d+(\.\d+)?$"), x.cast("double"))


def validar_tabela(nome, colunas_obrigatorias):
    df = spark.table(nome)
    faltantes = [c for c in colunas_obrigatorias if c not in df.columns]
    if faltantes:
        raise RuntimeError(f"{nome}: colunas ausentes: {faltantes}")
    print(f"OK - {nome}: {df.count()} linhas")


## 1) silver.tb_avaliacoes_usuarios


In [0]:
reviews = spark.table("bronze.tb_movies_reviews")

nota = numero_decimal(f.col("nota"))
comentario = f.trim(f.col("comentario"))

avaliacoes = (
    reviews.select(
        f.trim(f.col("id")).alias("id_filme"),
        f.trim(f.col("nome")).alias("nome_usuario"),
        f.when(nota.between(0, 10), nota).alias("nota_usuario"),
        f.when(
            comentario.isNull() | (comentario == ""),
            f.lit("Sem comentário")
        ).otherwise(comentario).alias("comentario_usuario"),
    )
    #Remocao de avaliacoes integralmente duplicadas pela combinacao de filme, usuario, nota e comentario
    .dropDuplicates(["id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"])
)

avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_avaliacoes_usuarios"
)

print("Notas fora de 0-10:", avaliacoes.filter((f.col("nota_usuario") < 0) | (f.col("nota_usuario") > 10)).count())
print("Comentarios vazios:", avaliacoes.filter(f.trim("comentario_usuario") == "").count())
avaliacoes.show(10, truncate=False)


Notas fora de 0-10: 0
Comentarios vazios: 0
+--------+---------------------+------------+-----------------------------------------------+
|id_filme|nome_usuario         |nota_usuario|comentario_usuario                             |
+--------+---------------------+------------+-----------------------------------------------+
|442113  |Mariana Cardoso 277  |4.4         |Sem comentário                                 |
|637007  |Lucas Reis 602       |3.9         |Sem comentário                                 |
|449479  |Sérgio Freitas       |0.7         |Péssimo em todos os sentidos.                  |
|413036  |Gabriela Monteiro 401|7.5         |Sem comentário                                 |
|528480  |Leonardo Monteiro    |6.3         |Assisti até o final mas não me marcou.         |
|387727  |Maria Alves 707      |8.2         |Gostei bastante, recomendo.                    |
|1032506 |Amanda Castro 242    |4.7         |Fraco, não recomendo.                          |
|446554  |Sandra

## 2) silver.tb_metricas_engajamento


In [0]:
metricas_raw = carga_mais_recente(spark.table("bronze.tb_movies_metrics"))

popularidade = numero_decimal(f.col("popularity"))
nota_tmdb = numero_decimal(f.col("vote_average"))
nota_imdb = numero_decimal(f.col("averageRating"))

#Contagens de votos sao convertidas somente quando representam valores inteiros validos
def inteiro_seguro(coluna):
    x = f.trim(coluna.cast("string"))
    x = f.regexp_replace(x, r"\s", "")
    x = f.regexp_replace(x, ",", "")  # separador de milhar em contagens
    return f.when(x.rlike(r"^-?\d+(\.0+)?$"), x.cast("double"))

votos_tmdb = inteiro_seguro(f.col("vote_count"))
votos_imdb = inteiro_seguro(f.col("numVotes"))

metricas = metricas_raw.select(
    f.col("id").alias("id_filme"),
    f.when(popularidade >= 0, popularidade).alias("popularidade"),
    f.when(nota_tmdb.between(0, 10), nota_tmdb).alias("nota_media_tmdb"),
    f.when(votos_tmdb >= 0, votos_tmdb).cast("int").alias("qtd_votos_tmdb"),
    f.when(nota_imdb.between(0, 10), nota_imdb).alias("nota_media_imdb"),
    f.when(votos_imdb >= 0, votos_imdb).cast("int").alias("qtd_votos_imdb"),
)

metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_metricas_engajamento"
)

metricas.select([f.count(f.when(f.col(c).isNull(), 1)).alias(c) for c in metricas.columns]).show()
metricas.show(10, truncate=False)


+--------+------------+---------------+--------------+---------------+--------------+
|id_filme|popularidade|nota_media_tmdb|qtd_votos_tmdb|nota_media_imdb|qtd_votos_imdb|
+--------+------------+---------------+--------------+---------------+--------------+
|       0|        4168|           3519|          8036|          13197|         11574|
+--------+------------+---------------+--------------+---------------+--------------+

+--------+------------+---------------+--------------+---------------+--------------+
|id_filme|popularidade|nota_media_tmdb|qtd_votos_tmdb|nota_media_imdb|qtd_votos_imdb|
+--------+------------+---------------+--------------+---------------+--------------+
|1000004 |1.132       |0.0            |0             |6.8            |27            |
|1000005 |0.6         |0.0            |0             |NULL           |40            |
|1000007 |0.6         |0.0            |0             |4.7            |10            |
|1000011 |1.169       |0.0            |0             

## 3) silver.tb_cotacao_dolar

Mantem uma cotacao por dia e cria serie diaria continua. Finais de semana/feriados recebem o ultimo valor util disponivel por Forward Fill.


In [0]:
cot_raw = spark.table("bronze.tb_cotacao_dolar")

cot = (
    cot_raw.select(
        f.to_timestamp("dataHoraCotacao").alias("data_hora_cotacao"),
        f.to_date(f.substring(f.col("dataHoraCotacao"), 1, 10), "yyyy-MM-dd").alias("data_cotacao"),
        f.col("cotacaoCompra").cast("double").alias("cotacao_compra"),
        f.col("ingestion_datetime"),
    )
    .filter(f.col("data_cotacao").isNotNull() & (f.col("cotacao_compra") > 0))
)

#Quando existem varias cotacoes no mesmo dia, permanece a cotacao mais recente do periodo
w_dia = Window.partitionBy("data_cotacao").orderBy(
    f.col("data_hora_cotacao").desc_nulls_last(),
    f.col("ingestion_datetime").desc()
)
cot = (
    cot.withColumn("_rn", f.row_number().over(w_dia))
       .filter(f.col("_rn") == 1)
       .select("data_cotacao", "cotacao_compra")
)

if cot.limit(1).count() == 0:
    raise RuntimeError("Nao ha cotacoes validas na Bronze.")

#Construcao de calendario diario para garantir continuidade da serie temporal de cotacao
#O calendario cobre o intervalo disponivel ate a data de referencia da execucao
#Dias sem cotacao recebem o ultimo valor valido por meio de forward fill
limites = cot.agg(
    f.min("data_cotacao").alias("inicio"),
    f.max("data_cotacao").alias("ultima_cotacao")
)

calendario = limites.select(
    f.explode(
        f.sequence(
            f.col("inicio"),
            f.greatest(f.col("ultima_cotacao"), f.current_date()),
            f.expr("INTERVAL 1 DAY")
        )
    ).alias("data_cotacao")
)

cotacao = calendario.join(cot, "data_cotacao", "left")
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
cotacao = cotacao.withColumn(
    "cotacao_compra",
    f.last("cotacao_compra", ignorenulls=True).over(w_ff)
)

cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_cotacao_dolar"
)

print("Dias sem cotacao apos Forward Fill:", cotacao.filter(f.col("cotacao_compra").isNull()).count())
cotacao.orderBy("data_cotacao").show(30, truncate=False)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dias sem cotacao apos Forward Fill: 0
+------------+--------------+
|data_cotacao|cotacao_compra|
+------------+--------------+
|2026-09-14  |5.169         |
|2026-09-15  |5.1484        |
|2026-09-16  |5.152         |
|2026-09-17  |5.1515        |
|2026-09-18  |5.1569        |
|2026-09-19  |5.1569        |
|2026-09-20  |5.1569        |
|2026-09-21  |5.1569        |
+------------+--------------+



## 4) silver.tb_financeiro_filmes


In [0]:
fin_raw = carga_mais_recente(spark.table("bronze.tb_movies_financials"))

orcamento_num = numero_monetario(f.col("budget"))
receita_num = numero_monetario(f.col("revenue"))

financeiro = fin_raw.select(
    f.col("id").alias("id_filme"),
    f.when(orcamento_num > 0, orcamento_num).cast("decimal(18,2)").alias("orcamento_usd"),
    f.when(receita_num > 0, receita_num).cast("decimal(18,2)").alias("receita_usd"),
)

#Utilizacao da cotacao mais recente disponivel na Silver para conversao dos valores financeiros para BRL
taxa_row = (
    spark.table("silver.tb_cotacao_dolar")
    .filter(f.col("cotacao_compra").isNotNull())
    .orderBy(f.col("data_cotacao").desc())
    .select("cotacao_compra")
    .first()
)
if taxa_row is None:
    raise RuntimeError("Nao foi possivel obter uma cotacao USD/BRL valida.")

taxa = float(taxa_row["cotacao_compra"])
print("Taxa USD -> BRL aplicada:", taxa)

financeiro = (
    financeiro
    .withColumn("orcamento_brl", (f.col("orcamento_usd") * f.lit(taxa)).cast("decimal(18,2)"))
    .withColumn("receita_brl", (f.col("receita_usd") * f.lit(taxa)).cast("decimal(18,2)"))
    .withColumn("lucro_usd", (f.col("receita_usd") - f.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (f.col("receita_brl") - f.col("orcamento_brl")).cast("decimal(18,2)"))
    .withColumn(
        "margem_lucro_pct",
        f.when(
            f.col("receita_usd") > 0,
            f.round((f.col("lucro_usd") / f.col("receita_usd")) * 100, 2)
        ).cast("decimal(10,2)")
    )
)

financeiro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_financeiro_filmes"
)

financeiro.printSchema()
financeiro.show(10, truncate=False)


Taxa USD -> BRL aplicada: 5.1569
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_pct: decimal(10,2) (nullable = true)

+--------+-------------+-----------+-------------+-----------+---------+---------+----------------+
|id_filme|orcamento_usd|receita_usd|orcamento_brl|receita_brl|lucro_usd|lucro_brl|margem_lucro_pct|
+--------+-------------+-----------+-------------+-----------+---------+---------+----------------+
|1000004 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL            |
|1000005 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL            |
|1000007 |NULL         |NULL       |NULL         |NULL       |NULL

## 5) silver.tb_info_filmes


In [0]:
info_raw = carga_mais_recente(spark.table("bronze.tb_movies_info"))

#Normalizacao e traducao do status dos filmes conforme regra de negocio
st = f.lower(f.trim(f.col("status")))
st = f.regexp_replace(st, r"[-_]+", " ")
st = f.regexp_replace(st, r"[^a-z ]", "")
st = f.trim(f.regexp_replace(st, r"\s+", " "))

status_filme = (
    f.when(st == "released", "Lançado")
     .when(st.isin("post production", "postproduction"), "Pós-Produção")
     .when(st.isin("in production", "inproduction"), "Em Produção")
     .when(st == "planned", "Planejado")
     .when(st == "rumored", "Rumores")
     .when(st.isin("canceled", "cancelled"), "Cancelado")
     .otherwise("Não Informado")
)

#Conversao robusta da data de lancamento considerando os formatos presentes na origem
raw_date = f.trim(f.col("release_date"))

#Formatos de data inequivocos sao avaliados antes dos formatos potencialmente ambiguos
data_iso = f.coalesce(
    f.expr("try_to_timestamp(trim(release_date), 'yyyy-MM-dd')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'yyyy-MM-dd HH:mm:ss')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'yyyy/M/d')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'yyyyMMdd')").cast("date"),
)

#Nos formatos com barras, os componentes de dia e mes sao avaliados para reduzir ambiguidades
#Quando os dois componentes sao validos como mes, utiliza-se uma regra deterministica de fallback
#para garantir consistencia durante o processamento
partes = f.split(f.regexp_replace(raw_date, "-", "/"), "/")
p1 = f.element_at(partes, 1).cast("int")
p2 = f.element_at(partes, 2).cast("int")

data_barra = (
    f.when(
        raw_date.rlike(r"^\d{1,2}[-/]\d{1,2}[-/]\d{4}$") & (p1 > 12),
        f.expr("try_to_timestamp(regexp_replace(trim(release_date), '-', '/'), 'd/M/yyyy')").cast("date")
    )
    .when(
        raw_date.rlike(r"^\d{1,2}[-/]\d{1,2}[-/]\d{4}$") & (p2 > 12),
        f.expr("try_to_timestamp(regexp_replace(trim(release_date), '-', '/'), 'M/d/yyyy')").cast("date")
    )
    .when(
        raw_date.rlike(r"^\d{1,2}[-/]\d{1,2}[-/]\d{4}$"),
        f.expr("try_to_timestamp(regexp_replace(trim(release_date), '-', '/'), 'M/d/yyyy')").cast("date")
    )
)

data_lancamento = f.coalesce(data_iso, data_barra)

#A duracao e mantida somente quando numerica e nao negativa
duracao_txt = f.trim(f.col("runtime"))
duracao = f.when(
    duracao_txt.rlike(r"^\d+(\.0+)?$"),
    duracao_txt.cast("double")
)

info = (
    info_raw.select(
        f.col("id").alias("id_filme"),
        f.trim(f.col("title")).alias("titulo"),
        f.trim(f.col("original_title")).alias("titulo_original"),
        data_lancamento.alias("data_lancamento"),
        f.when(duracao >= 0, duracao).cast("int").alias("duracao_minutos"),
        f.trim(f.col("original_language")).alias("idioma_original"),
        status_filme.alias("status_filme"),
        f.col("overview").alias("sinopse"),
        f.col("tagline").alias("frase_divulgacao"),
    )
    .withColumn("ano_lancamento", f.year("data_lancamento"))
)

info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_info_filmes"
)

print("Filmes:", info.count(), "| IDs distintos:", info.select("id_filme").distinct().count())
info.groupBy("status_filme").count().orderBy(f.desc("count")).show()
info.show(10, truncate=False)


Filmes: 97879 | IDs distintos: 97879
+-------------+-----+
| status_filme|count|
+-------------+-----+
|      Lançado|96463|
| Pós-Produção|  701|
|  Em Produção|  604|
|Não Informado|   64|
|    Planejado|   47|
+-------------+-----+

+--------+------------------------------------------+------------------------------------------+---------------+---------------+---------------+------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 6) silver.tb_generos

O dominio de generos cinematograficos e controlado. Apos split + explode, somente valores pertencentes ao conjunto valido sao mantidos. Isso remove residuos de Column Shift, paises, idiomas, nomes e frases deslocadas.


In [0]:
credits_raw = carga_mais_recente(spark.table("bronze.tb_credits_and_tags"))

#Definicao do dominio valido de generos para eliminar residuos causados por deslocamento de colunas
generos_validos = {
    "action": "Action",
    "adventure": "Adventure",
    "animation": "Animation",
    "comedy": "Comedy",
    "crime": "Crime",
    "documentary": "Documentary",
    "drama": "Drama",
    "family": "Family",
    "fantasy": "Fantasy",
    "history": "History",
    "horror": "Horror",
    "music": "Music",
    "mystery": "Mystery",
    "romance": "Romance",
    "science fiction": "Science Fiction",
    "tv movie": "TV Movie",
    "thriller": "Thriller",
    "war": "War",
    "western": "Western",
}

#Padronizacao dos separadores e remocao de caracteres residuais antes da explosao dos generos
genero_bruto = f.explode(
    f.split(f.regexp_replace(f.col("genres"), r"[;|]", ","), ",")
)

generos = credits_raw.select(
    f.col("id").alias("id_filme"),
    genero_bruto.alias("_genero")
)

generos = generos.withColumn(
    "_genero_norm",
    f.lower(
        f.trim(
            f.regexp_replace(
                f.regexp_replace(f.col("_genero"), r'^[\s\[\]\{\}\"]+|[\s\[\]\{\}\"]+$', ""),
                r"\s+", " "
            )
        )
    )
)

#Mapeamento realizado com funcoes nativas do Spark para manter a otimizacao do plano de execucao
map_expr = f.create_map(*sum(([f.lit(k), f.lit(v)] for k, v in generos_validos.items()), []))

generos = (
    generos.withColumn("nome_genero", map_expr[f.col("_genero_norm")])
           .filter(f.col("nome_genero").isNotNull())
           .select("id_filme", "nome_genero")
           .dropDuplicates()
)

generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_generos"
)

generos.groupBy("nome_genero").count().orderBy(f.desc("count"), "nome_genero").show(50, truncate=False)
print("Quantidade de generos invalidos na Silver:", generos.filter(~f.col("nome_genero").isin(list(generos_validos.values()))).count())


+---------------+-----+
|nome_genero    |count|
+---------------+-----+
|Drama          |32682|
|Documentary    |19248|
|Comedy         |18860|
|Thriller       |10410|
|Horror         |9868 |
|Romance        |7723 |
|Action         |6114 |
|Crime          |4795 |
|Animation      |4528 |
|TV Movie       |4121 |
|Science Fiction|3811 |
|Family         |3770 |
|Mystery        |3356 |
|Fantasy        |3310 |
|Adventure      |2893 |
|Music          |2826 |
|History        |2443 |
|War            |974  |
|Western        |417  |
+---------------+-----+

Quantidade de generos invalidos na Silver: 0


## 7) silver.tb_pessoas_empresas

Consolida cast, directors, writers e production_companies. Alem da limpeza estrutural, remove marcadores de ausencia e termos de outros dominios que aparecem por Column Shift.


In [0]:
#Definicao de termos invalidos para impedir a permanencia de valores deslocados nas entidades
#A lista contempla marcadores de ausencia, status e valores pertencentes a outros dominios
termos_invalidos = {
    "", "n/a", "na", "null", "none", "unknown", "nao informado", "nao informado", "[]", "{}",
    "released", "planned", "rumored", "canceled", "cancelled", "post production", "in production",
    "english", "portuguese", "spanish", "french", "german", "italian", "japanese", "korean",
    "mandarin", "cantonese", "russian", "arabic", "hindi", "xhosa", "romanian",
    "united states of america", "united states", "united kingdom", "canada", "italy", "france",
    "germany", "spain", "brazil", "japan", "china", "india", "australia", "mexico",
    "however", "of course", "in fact", "deleted scenes",
} | set(generos_validos.keys())


def explodir_entidades(df, coluna_origem, tipo_entidade, permitir_digitos=False, max_palavras=8):
    itens = f.split(f.regexp_replace(f.col(coluna_origem), r"[;|]", ","), ",")
    out = df.select(
        f.col("id").alias("id_filme"),
        f.explode(itens).alias("_nome")
    )

    #Remocao de pontuacao residual nas extremidades e normalizacao de espacos
    out = out.withColumn(
        "_nome",
        f.trim(
            f.regexp_replace(
                f.regexp_replace(f.col("_nome"), r'^[\s\[\]\{\}\"]+|[\s\[\]\{\}\"]+$', ""),
                r"\s+", " "
            )
        )
    )
    out = out.withColumn("_nome_lower", f.lower("_nome"))

    filtro = (
        f.col("_nome").isNotNull()
        & (f.length("_nome") >= 2)
        & (f.length("_nome") <= 100)
        & (f.size(f.split(f.col("_nome"), r"\s+")) <= max_palavras)
        & (~f.col("_nome_lower").isin(list(termos_invalidos)))
        & (~f.col("_nome").rlike(r"[\[\]{}]"))
        #Textos com caracteristicas de frases sao descartados por nao representarem entidades validas
        & (~f.col("_nome").rlike(r"[!?]"))
    )

    if not permitir_digitos:
        filtro = filtro & (~f.col("_nome").rlike(r"\d"))
    else:
        #Nomes de produtoras podem conter numeros, mas valores exclusivamente numericos sao descartados
        filtro = filtro & (~f.col("_nome").rlike(r"^[0-9\W_]+$"))

    out = out.filter(filtro)

    #Padronizacao da capitalizacao textual das entidades
    out = out.select(
        "id_filme",
        f.initcap(f.col("_nome")).alias("nome_entidade"),
        f.lit(tipo_entidade).alias("tipo_entidade")
    )

    return out.dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])

atores = explodir_entidades(credits_raw, "cast", "Ator", permitir_digitos=False, max_palavras=8)
diretores = explodir_entidades(credits_raw, "directors", "Diretor", permitir_digitos=False, max_palavras=8)
roteiristas = explodir_entidades(credits_raw, "writers", "Roteirista", permitir_digitos=False, max_palavras=8)
produtoras = explodir_entidades(credits_raw, "production_companies", "Produtora", permitir_digitos=True, max_palavras=10)

pessoas_empresas = (
    atores.unionByName(diretores)
          .unionByName(roteiristas)
          .unionByName(produtoras)
          .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "silver.tb_pessoas_empresas"
)

pessoas_empresas.groupBy("tipo_entidade").count().show()
pessoas_empresas.show(20, truncate=False)


+-------------+------+
|tipo_entidade| count|
+-------------+------+
|      Diretor|100496|
|   Roteirista|125573|
|    Produtora|117532|
|         Ator|541368|
+-------------+------+

+--------+--------------------+-------------+
|id_filme|nome_entidade       |tipo_entidade|
+--------+--------------------+-------------+
|1000172 |Abdellah El Hajjouji|Ator         |
|1000475 |Nicholas Wittman    |Ator         |
|1001178 |James O'connell     |Ator         |
|1001840 |Toni Laudadio       |Ator         |
|1002079 |Aju Varghese        |Ator         |
|1002401 |Karla Horvat        |Ator         |
|1002482 |Ivan Massagué       |Ator         |
|1003448 |Ana Takač           |Ator         |
|1004269 |Melinda Dekay       |Ator         |
|1004555 |August Diehl        |Ator         |
|1005508 |Nathalie Hambro     |Ator         |
|1005942 |David Cherta        |Ator         |
|1005972 |Moa Nilsson         |Ator         |
|1006318 |Sophie Alakija      |Ator         |
|1006717 |Hardik Sangani      |At

## 8) Validacoes finais da camada Silver


In [0]:
validar_tabela("silver.tb_info_filmes", [
    "id_filme", "titulo", "titulo_original", "data_lancamento", "duracao_minutos",
    "idioma_original", "status_filme", "sinopse", "frase_divulgacao", "ano_lancamento"
])
validar_tabela("silver.tb_financeiro_filmes", [
    "id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
    "lucro_usd", "lucro_brl", "margem_lucro_pct"
])
validar_tabela("silver.tb_metricas_engajamento", [
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
])
validar_tabela("silver.tb_avaliacoes_usuarios", [
    "id_filme", "nome_usuario", "nota_usuario", "comentario_usuario"
])
validar_tabela("silver.tb_generos", ["id_filme", "nome_genero"])
validar_tabela("silver.tb_pessoas_empresas", ["id_filme", "nome_entidade", "tipo_entidade"])
validar_tabela("silver.tb_cotacao_dolar", ["data_cotacao", "cotacao_compra"])

#Validacoes finais de qualidade das tabelas geradas na camada Silver
assert spark.table("silver.tb_info_filmes").groupBy("id_filme").count().filter(f.col("count") > 1).count() == 0
assert spark.table("silver.tb_metricas_engajamento").filter(
    (f.col("nota_media_tmdb") < 0) | (f.col("nota_media_tmdb") > 10) |
    (f.col("nota_media_imdb") < 0) | (f.col("nota_media_imdb") > 10) |
    (f.col("popularidade") < 0) |
    (f.col("qtd_votos_tmdb") < 0) |
    (f.col("qtd_votos_imdb") < 0)
).count() == 0
assert spark.table("silver.tb_avaliacoes_usuarios").filter(
    (f.col("nota_usuario") < 0) | (f.col("nota_usuario") > 10)
).count() == 0
assert spark.table("silver.tb_cotacao_dolar").filter(f.col("cotacao_compra").isNull()).count() == 0
assert spark.table("silver.tb_generos").filter(
    ~f.col("nome_genero").isin(list(generos_validos.values()))
).count() == 0

print("Bronze_to_Silver concluido com sucesso.")


OK - silver.tb_info_filmes: 97879 linhas
OK - silver.tb_financeiro_filmes: 99006 linhas
OK - silver.tb_metricas_engajamento: 99013 linhas
OK - silver.tb_avaliacoes_usuarios: 32412 linhas
OK - silver.tb_generos: 142149 linhas
OK - silver.tb_pessoas_empresas: 884969 linhas
OK - silver.tb_cotacao_dolar: 8 linhas
Bronze_to_Silver concluido com sucesso.
